# 01 — Problématique, dataset et préparation des données

**Projet :** Fine-tuning d'un petit LLM avec LoRA pour générer des requêtes SQL sur des bases financières.

Ce notebook couvre les étapes 1 à 4 de la méthodologie :
1. Problématique
2. État de l'art
3. Dataset
4. Nettoyage et préparation

## 1. Problématique

Les équipes métier d'une institution financière (analystes, contrôleurs, managers) ont besoin d'interroger des bases SQL mais ne savent pas écrire de SQL. Elles dépendent de l'équipe data, ce qui crée des délais. Les grands LLM propriétaires savent traduire une question en SQL, mais une banque ne peut pas toujours leur envoyer ses schémas et ses questions pour des raisons de confidentialité et de coût.

> **Question du projet :** un petit modèle open source, fine-tuné avec LoRA sur un seul GPU, peut-il générer des requêtes SQL fiables sur des bases financières, tout en restant déployable en interne ?

## 2. État de l'art

Plusieurs travaux ont étudié la génération de requêtes SQL dans la finance. **BIRD** (2023) montre que les LLM peinent sur des bases réelles, où seuls les très gros modèles atteignent environ 80 %. **BookSQL** (2024) confirme que même GPT-4 échoue souvent en comptabilité. **FinSQL** (2024) montre que LoRA permet d'adapter un LLM à la finance à faible coût.

- BIRD : https://arxiv.org/abs/2305.03111
- BookSQL : https://aclanthology.org/2024.naacl-long.28
- FinSQL : https://arxiv.org/abs/2401.10506

## 0. Installation

À exécuter au début de chaque session Colab. 

In [ ]:
import os, sys

REPO_URL = "https://github.com/eya-bouhmida/text2sql-finance-qlora.git"
REPO_DIR = "/content/text2sql-finance-qlora"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull
    %cd {REPO_DIR}
    !pip install -q datasets pandas matplotlib
    sys.path.append(REPO_DIR)
else:
    sys.path.append("..")  # exécution locale depuis le dossier notebooks/

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from src import data as D

pd.set_option("display.max_colwidth", 120)

## 3. Dataset

**Gretel `synthetic_text_to_sql`** (licence Apache 2.0) : environ 105 000 exemples synthétiques (100 000 train, 5 851 test) couvrant 100 domaines. Chaque exemple contient une question, le contexte SQL (`CREATE TABLE` + `INSERT`) et la requête attendue.

Comme le contexte contient des données, on peut **créer une mini-base SQLite et exécuter les requêtes**, ce qui permettra d'évaluer le modèle par *execution accuracy*.

In [ ]:
train_raw, test_raw = D.load_gretel()
print(f"Train : {len(train_raw):,} exemples | Test : {len(test_raw):,} exemples")
print("Colonnes :", list(train_raw.columns))

In [ ]:
# Un exemple complet
ex = train_raw.iloc[0]
for col in ["domain", "sql_complexity", "sql_task_type", "sql_prompt", "sql_context", "sql"]:
    print(f"--- {col} ---\n{ex[col]}\n")

### 3.1 Exploration du dataset complet

In [ ]:
print("Nombre de domaines :", train_raw["domain"].nunique())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
train_raw["sql_complexity"].value_counts().sort_values().plot.barh(ax=axes[0], color="#4C72B0")
axes[0].set_title("Complexité des requêtes")
train_raw["sql_task_type"].value_counts().sort_values().plot.barh(ax=axes[1], color="#4C72B0")
axes[1].set_title("Type de tâche")
plt.tight_layout(); plt.show()

### 3.2 Sélection des domaines financiers

On repère les domaines dont le nom contient un mot-clé financier (`finance`, `bank`, `fintech`, `invest`, `insurance`…).


In [ ]:
all_domains = sorted(train_raw["domain"].unique())
print(all_domains)

In [ ]:
FINANCE_DOMAINS = D.find_finance_domains(all_domains)
# Ajustements manuels si besoin :
# FINANCE_DOMAINS += ["..."]
# FINANCE_DOMAINS.remove("...")
print("Domaines retenus :", FINANCE_DOMAINS)

train_fin = D.filter_domains(train_raw, FINANCE_DOMAINS)
test_fin = D.filter_domains(test_raw, FINANCE_DOMAINS)
print(f"Train finance : {len(train_fin):,} | Test finance : {len(test_fin):,}")

train_fin["domain"].value_counts().sort_values().plot.barh(figsize=(7, 3), color="#4C72B0",
                                                            title="Exemples par domaine financier")
plt.show()

## 4. Nettoyage et préparation

| Étape | Pourquoi |
|---|---|
| Garder les tâches en lecture seule | Le cas d'usage est un analyste qui **interroge** une base, pas qui la modifie (pas d'`INSERT`, `UPDATE`, `DELETE`) |
| Garder les requêtes `SELECT` / `WITH` | Sécurité : le modèle ne doit jamais apprendre à modifier les données |
| Supprimer les doublons | Éviter que le modèle mémorise, et éviter les fuites entre train et test |
| Vérifier l'exécution dans SQLite | Certaines requêtes utilisent une syntaxe MySQL ou PostgreSQL : on ne garde que ce qu'on peut évaluer |

In [ ]:
train_clean, train_report, train_errors = D.clean_dataset(train_fin)
test_clean, test_report, test_errors = D.clean_dataset(test_fin, require_non_empty=True)

report = pd.DataFrame({"Train": train_report, "Test": test_report})
report

Pour le **test**, on retire aussi les requêtes qui renvoient un résultat vide : sinon, n'importe quelle requête qui ne renvoie rien serait comptée comme correcte, ce qui gonflerait artificiellement l'execution accuracy.

In [ ]:
# Les erreurs d'exécution les plus fréquentes (requêtes écartées)
train_errors.str.split(":").str[0:2].str.join(":").value_counts().head(10)

### 4.1 Séparation entraînement / validation

In [ ]:
from sklearn.model_selection import train_test_split

# Stratification par complexité (si chaque classe a au moins 2 exemples)
counts = train_clean["sql_complexity"].value_counts()
strat = train_clean["sql_complexity"] if counts.min() >= 2 else None
train_df, val_df = train_test_split(train_clean, test_size=0.1, random_state=42, stratify=strat)

# Éviter toute fuite : on retire du train et de la validation les questions présentes dans le test
test_questions = set(test_clean["sql_prompt"])
n_overlap = train_df["sql_prompt"].isin(test_questions).sum() + val_df["sql_prompt"].isin(test_questions).sum()
train_df = train_df[~train_df["sql_prompt"].isin(test_questions)]
val_df = val_df[~val_df["sql_prompt"].isin(test_questions)]

print("Questions communes avec le test, retirées :", n_overlap)
print(f"Train : {len(train_df):,} | Validation : {len(val_df):,} | Test : {len(test_clean):,}")

### 4.2 Statistiques après nettoyage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
pd.DataFrame({
    "Train": train_df["sql_complexity"].value_counts(normalize=True),
    "Test": test_clean["sql_complexity"].value_counts(normalize=True),
}).sort_values("Train").plot.barh(ax=axes[0])
axes[0].set_title("Complexité (proportion)")

lengths = train_df["sql_context"].apply(D.schema_only).str.len() + train_df["sql_prompt"].str.len()
lengths.plot.hist(bins=40, ax=axes[1], color="#4C72B0")
axes[1].set_title("Longueur du prompt (caractères)")
plt.tight_layout(); plt.show()

print(f"Longueur médiane : {lengths.median():.0f} caractères | 95e percentile : {lengths.quantile(0.95):.0f}")

La longueur en caractères donne une première idée ; la longueur réelle en tokens sera vérifiée avec le tokenizer du modèle dans le notebook 03.

### 4.3 Format des prompts

Le modèle reçoit **uniquement le schéma** (`CREATE TABLE`), jamais les données (`INSERT`). C'est le fonctionnement réel d'un assistant Text-to-SQL : les données restent dans la base, et c'est notre code qui exécute la requête générée.

In [ ]:
train_records = D.to_records(train_df)
val_records = D.to_records(val_df)
test_records = D.to_records(test_clean, with_answer=False)

for m in train_records[0]["messages"]:
    print(f"[{m['role'].upper()}]\n{m['content']}\n")

### 4.4 Sauvegarde

In [ ]:
DATA_DIR = "/content/drive/MyDrive/text2sql-finance-qlora/data" if IN_COLAB else "../data"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

D.save_jsonl(train_records, f"{DATA_DIR}/train.jsonl")
D.save_jsonl(val_records, f"{DATA_DIR}/val.jsonl")
D.save_jsonl(test_records, f"{DATA_DIR}/test.jsonl")

# Le rapport de nettoyage va dans results/ (versionné sur GitHub)
RESULTS_DIR = f"{REPO_DIR}/results" if IN_COLAB else "../results"
os.makedirs(RESULTS_DIR, exist_ok=True)
report.to_csv(f"{RESULTS_DIR}/01_cleaning_report.csv")
print("Données sauvegardées dans", DATA_DIR)

## Résumé

- Dataset : Gretel `synthetic_text_to_sql`, filtré sur les domaines financiers.
- Nettoyage : lecture seule, `SELECT` uniquement, doublons supprimés, exécution vérifiée dans SQLite.
- Sorties : `train.jsonl`, `val.jsonl`, `test.jsonl` (sur Drive) et le rapport de nettoyage (`results/`).

**Prochaine étape :** notebook 02, mesurer la performance du modèle **avant** fine-tuning (baseline).